# Results of the feedback-loop experiments

This notebook only reads `results/` and plots. All simulation code is in `src/refugee_sim/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from refugee_sim.generator import STATE_QUOTA, normalise
from refugee_sim.simulation import load_results

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})

# results come only from results/ (python -m refugee_sim.simulation configs/main.yaml configs/history.yaml configs/forms.yaml)
DELTAS = [-0.3, -0.15, 0.0, 0.15, 0.3]
grid = pd.concat([load_results(f"../results/{name}") for name in ["main", "history", "forms"]], ignore_index=True)
main = grid[(grid["form"] == "linear") & (grid["n_hist"] == 10000)]
print(f"{grid['run_key'].nunique()} runs loaded")

## 7. Results

### 7.1 True gain over random quota placement

In [ ]:
colors = {-0.3: "#B2182B", -0.15: "#EF8A62", 0.0: "#555555", 0.15: "#67A9CF", 0.3: "#2166AC"}
scen = ["none", "SOEP", "stylised x4"]; pols = ["fixed", "every_3", "every_round"]
fig, axes = plt.subplots(3, 3, figsize=(16, 11), sharex=True)
for i, s in enumerate(scen):
    for j, p in enumerate(pols):
        ax = axes[i, j]
        m_ = main[(main["interactions"] == s) & (main["policy"] == p)].groupby(["delta", "round"])["gain_pct"].mean().unstack(0)
        o_ = main[(main["interactions"] == s) & (main["policy"] == p) & (main["delta"] == 0)].groupby("round")["oracle_gain_pct"].mean()
        for d in DELTAS:
            ax.plot(m_.index, m_[d], "o-", ms=3, color=colors[d], label=f"δ = {d:+.2f}")
        ax.plot(o_.index, o_, ":", color="k", label="best possible (δ = 0)")
        ax.axhline(0, color="k", lw=0.5)
        ax.set_title(f"{s} interactions · {p}", fontsize=10, fontweight="bold")
        if j == 0: ax.set_ylabel("true gain (%)")
        if i == 2: ax.set_xlabel("round")
axes[0, 0].legend(fontsize=7, frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
summary = (main[main["round"].isin([1, 10])]
           .groupby(["interactions", "delta", "policy", "round"])[["gain_pct", "oracle_gain_pct", "captured_pct", "promised_gain_pct"]]
           .mean().round(1).unstack("round"))
summary

### 7.2 What the model promises vs. what really happens

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
for ax, s in zip(axes, scen):
    for d, ls in ((-0.3, "-"), (0.0, "--"), (0.3, ":")):
        m_ = main[(main["interactions"] == s) & (main["policy"] == "fixed") & (main["delta"] == d)].groupby("round")[["gain_pct", "promised_gain_pct"]].mean()
        ax.plot(m_.index, m_["promised_gain_pct"], ls, color="#7B3FA0", label=f"promised, δ={d:+.1f}")
        ax.plot(m_.index, m_["gain_pct"], ls, color="k", label=f"true, δ={d:+.1f}")
    ax.axhline(0, color="grey", lw=0.5); ax.set_title(f"{s} interactions · fixed model", fontweight="bold"); ax.set_xlabel("round")
axes[0].set_ylabel("gain over quota (%)"); axes[0].legend(fontsize=7, frameon=False)
plt.tight_layout(); plt.show()

### 7.3 How much history does the model need? (δ = 0, retrained every round)

In [ ]:
hist = grid[(grid["form"] == "linear") & (grid["delta"] == 0) & (grid["policy"] == "every_round") & grid["interactions"].isin(["SOEP", "stylised x4"])]
h = hist.groupby(["interactions", "n_hist"])[["captured_pct", "gain_pct", "oracle_gain_pct"]].mean().round(1)
fig, ax = plt.subplots(figsize=(7, 4))
for s, c in (("SOEP", "#2F6FB5"), ("stylised x4", "#E07B28")):
    x = h.loc[s]; ax.plot(x.index, x["captured_pct"], "o-", color=c, label=s)
ax.set_xscale("log"); ax.set_xlabel("historical arrivals used for the first model"); ax.set_ylabel("share of best possible gain captured (%)")
ax.legend(frameon=False); plt.tight_layout(); plt.show()
h

### 7.4 Other shapes of the community effect (SOEP interactions)

In [ ]:
shp = grid[(grid["interactions"] == "SOEP") & (grid["n_hist"] == 10000) & ((grid["form"] != "linear") | grid["delta"].isin([-0.3, 0.0, 0.3])) & grid["policy"].isin(["fixed", "every_round"])].copy()
shp["model"] = shp["form"] + " " + shp[["delta", "kappa", "delta_female", "delta_recent"]].fillna(0).astype(str).agg("/".join, axis=1)
t = shp[shp["round"].isin([1, 10])].groupby(["model", "policy", "round"])["gain_pct"].mean().round(1).unstack("round")
t["change"] = (t[10] - t[1]).round(1)
t

### 7.5 Where are people placed? Concentration of Syrian arrivals

In [ ]:
syr = [c for c in grid.columns if c.startswith("place__") and c.endswith("__Syria")]
def neff(row):
    x = row[syr].to_numpy(float); p = x / x.sum(); return 1 / np.sum(p ** 2)
quota_neff = 1 / np.sum(normalise(STATE_QUOTA) ** 2)
sub = main[(main["policy"] == "fixed") & main["delta"].isin([-0.3, 0.0, 0.3])].copy()
sub["neff_syria"] = sub.apply(neff, axis=1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, s in zip(axes, scen):
    m_ = sub[sub["interactions"] == s].groupby(["delta", "round"])["neff_syria"].mean().unstack(0)
    for d in m_.columns: ax.plot(m_.index, m_[d], "o-", ms=3, color=colors[d], label=f"δ = {d:+.1f}")
    ax.axhline(quota_neff, color="grey", ls="--", label="random quota placement")
    ax.set_title(f"{s} interactions · fixed model", fontweight="bold"); ax.set_xlabel("round")
axes[0].set_ylabel("effective number of states\nreceiving Syrians"); axes[0].legend(fontsize=7, frameon=False)
plt.tight_layout(); plt.show()

### 7.6 Who is affected (round 10)

In [ ]:
main[main["round"] == 10].groupby(["interactions", "delta", "policy"])[["origin_gap", "sex_gap", "mean_z_placed"]].mean().round(3).unstack("policy")